Calculate the Gamma-point frequency spectrum (195–225 THz) for the structure
in `test-str.ipynb` using Tidy3d MCP. Matching dipoles are added for the
hexagonal supercell.

In [ ]:
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import tidy3d as td
from tidy3d import web

# Reproducible randomness for source/monitor placement
np.random.seed(12)

02:03:10 Pacific Daylight Time WARNING: Using canonical configuration directory 
                               at 'C:\Users\xtliang\.config\tidy3d'. Found      
                               legacy directory at '~/.tidy3d', which will be   
                               ignored. Remove it manually or run 'tidy3d config
                               migrate --delete-legacy' to clean up.            

In [ ]:
# Copy of slab_hex_array from test-str.ipynb (creates hexagonal hole arrays)
def slab_hex_array(
	x0,
	y0,
	z0,
	R,
	hole_spacing_x,
	hole_spacing_y,
	n_x,
	n_y,
	height,
	hole_medium,
	slab_medium,
	reference_plane="bottom",
	sidewall_angle=0,
	axis=2,
	rotation=0,
):
	x_slab_length, y_slab_length = hole_spacing_x * (n_x + 0.5), hole_spacing_y * n_y

	i0 = n_x // 2
	j0 = n_y // 2

	start_x = x0 - (i0 + (j0 % 2) * 0.5) * hole_spacing_x
	start_y = y0 - j0 * hole_spacing_y

	box = td.Box(center=(x0, y0, z0), size=(x_slab_length, y_slab_length, height))

	structures = [td.Structure(geometry=box, medium=slab_medium)]

	cylinders = []

	cos_theta = np.cos(rotation)
	sin_theta = np.sin(rotation)

	for i in range(0, n_x):
		for j in range(0, n_y):
			cx = start_x + (i + (j % 2) * 0.5) * hole_spacing_x
			cy = start_y + j * hole_spacing_y
			dx = cx - x0
			dy = cy - y0
			rx = x0 + cos_theta * dx - sin_theta * dy
			ry = y0 + sin_theta * dx + cos_theta * dy

			c = td.Cylinder(
				axis=axis,
				sidewall_angle=sidewall_angle,
				reference_plane=reference_plane,
				radius=R,
				center=(rx, ry, z0),
				length=height,
			)
			cylinders.append(c)

	cylinders_structure = td.Structure(
		geometry=td.GeometryGroup(geometries=cylinders), medium=hole_medium
	)

	structures.append(cylinders_structure)

	return structures

# Parameters (taken from test-str.ipynb)
theta = 9.43 * np.pi / 180
am = 3.04

# Create the two rotated arrays and combine into the moire structure
s1 = slab_hex_array(
	0,
	0,
	0,
	0.12,
	0.5,
	0.5 * np.sqrt(3) / 2,
	30,
	30,
	0.2,
	td.Medium(permittivity=1**2),
	td.Medium(permittivity=3.4**2),
	rotation=np.pi / 6 + theta / 2,
)

s2 = slab_hex_array(
	0,
	0,
	0,
	0.12,
	0.5,
	0.5 * np.sqrt(3) / 2,
	30,
	30,
	0.2,
	td.Medium(permittivity=1**2),
	td.Medium(permittivity=3.4**2),
	rotation=np.pi / 6 - theta / 2,
)

moire_structures = [s1[0], s1[1], s2[1]]

In [ ]:
# Simulation / source / monitor settings for Gamma-point spectral run
polarization = "Ez"

# Frequency window requested by user
f_min = 195e12
f_max = 225e12
f_center = 0.5 * (f_min + f_max)
f_width = (f_max - f_min)

runTime = 2e-12

# size: use one supercell similar to test-str
size = [am, am * np.sqrt(3), 10]

# Grid spec: pick center wavelength ~ c/f_center in microns
wavelength_um = (td.constants.C_0 / f_center) / 1e-6
grid_spec = td.GridSpec.auto(wavelength=wavelength_um)

# Create monitors and sources (randomized positions within the supercell)
Nsources = 5
Nmonitors = 5

sourceTime = [
	td.GaussianPulse(freq0=f_center, fwidth=f_width, phase=i)
	for i in 2 * np.pi * np.random.random(2 * Nsources)
]

posySource = np.random.uniform(-0.8 * am, -0.2 * am, Nsources)
posxSource = np.random.uniform(-0.2 * am, 0.2 * am, Nsources)

posyMon = np.random.uniform(-0.8 * am, 0.8 * am, Nmonitors)
posxMon = np.random.uniform(-0.2 * am, 0.2 * am, Nmonitors)

monitors = [
	td.FieldTimeMonitor(center=(posxMon[i], posyMon[i], 0), name=f"mon{i}", size=(0, 0, 0), start=0)
	for i in range(Nmonitors)
]

# Function to build simulation with matching dipoles for hexagonal supercell
def getSim_gamma(pol, bloch_x=0, bloch_y=0, matchingDipoles=True):
	symmetry = [0, 0, -1] if pol == "Ez" else [0, 0, 1]

	boundary_spec = td.BoundarySpec(
		x=td.Boundary.bloch(bloch_x * size[0]),
		y=td.Boundary.bloch(bloch_y * size[1]),
		z=td.Boundary.pml(),
	)

	bVector = np.array([bloch_x, bloch_y, 0])

	# Create primary sources
	sources = [
		td.PointDipole(center=(posxSource[i], posySource[i], 0), polarization=pol, source_time=sourceTime[i])
		for i in range(len(posxSource))
	]

	# matching dipoles: shift by the primitive vector (in units consistent with structure)
	matching_dipoles = []
	if matchingDipoles:
		# primitive shift in same units as positions (approximate for this supercell)
		px_shift = 0.5 * am
		py_shift = (np.sqrt(3) / 2) * am
		for i, sc in enumerate(sources):
			px = sc.center[0] + px_shift
			py = sc.center[1] + py_shift

			phase = sc.source_time.phase
			r_vec = np.array((px, py, 0)) - np.array(sc.center)
			deltaPhase = 2 * np.pi * bVector.dot(r_vec / am)  # normalized to lattice units

			source_time = sc.source_time.updated_copy(phase=phase + deltaPhase)

			sp = sc.updated_copy(center=(px, py, 0), source_time=source_time)
			matching_dipoles.append(sp)

	sim = td.Simulation(
		size=size,
		structures=moire_structures,
		sources=sources + matching_dipoles,
		monitors=monitors,
		run_time=runTime,
		boundary_spec=boundary_spec,
		grid_spec=grid_spec,
		symmetry=symmetry,
		medium=td.Medium(),
	)
	return sim

In [ ]:
# Build and run a single Gamma-point simulation via Tidy3d MCP (web.Batch)
sims = {"gammaEz": getSim_gamma("Ez", 0, 0, matchingDipoles=True)}

batch = web.Batch(simulations=sims, folder_name="gamma_point_spectrum")
batch_data = batch.run(path_dir="gamma_spectrum")

# Retrieve simulation data
sim_data = batch_data["gammaEz"]

# Sum monitors' time traces
def getSignal(sim_data, polarization, t_start=0, t_end=runTime):
    signal = 0
    for i in range(Nmonitors):
        signal += sim_data[f"mon{i}"].field_components[polarization].squeeze()
    return signal.where(signal.t > t_start, drop=True).where(signal.t < t_end, drop=True)

signal = getSignal(sim_data, polarization)

# Convert to numpy array and compute FFT
try:
    sig = signal.values
except Exception:
    sig = np.array(signal)

# If the signal has multiple components (e.g., complex), take magnitude
if np.iscomplexobj(sig):
    sig = np.abs(sig)

# Ensure 1D
sig = sig.ravel()

# Time step
dt = sim_data.simulation.dt
N = len(sig)

freqs = np.fft.rfftfreq(N, d=dt)
amps = np.abs(np.fft.rfft(sig))

# Select user requested window and plot
mask = (freqs >= f_min) & (freqs <= f_max)
plt.figure(figsize=(6,4))
plt.plot(freqs[mask] / 1e12, amps[mask])
plt.xlabel("Frequency (THz)")
plt.ylabel("Amplitude (a.u.)")
plt.title("Gamma-point spectrum 195–225 THz")
plt.show()
